**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/pelot-2020"], check=True)
    os.chdir("/content/encore/entries/pelot-2020")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings of pelot-2020-01, checked on the versions of pelot-2020-02

This notebook belongs to the reconcile step of attempt pelot-2020-02 (docs/protocol.md, "Repeat
attempts"). It checks the findings of the earlier attempt, pelot-2020-01 (made on sparc-60
version 5), that pelot-2020-02 did not report, on the versions pelot-2020-02 pins: sparc-60
version 5, sparc-64 version 4, sparc-65 version 8, sparc-89 version 1, sparc-16 version 5, sparc-61
version 3 and sparc-82 version 1. Both findings checked here are about sparc-60. The files are read
with `pelot.py`, at the object versions of each version's manifest.

Earlier findings 1, 3, 5, 6, 7, 8 and 9 are reported again by pelot-2020-02 (its findings 1, 5,
11, 9 and 10) and are not checked here.

In [ ]:
import numpy as np
import pandas as pd
import pelot

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Earlier finding 2

> The code folder's RatVNMorphology_OutputMetrics.mat keeps only 7 of the 22 rat perineurium
> thicknesses (thkperi_um is overwritten per sex and level category in DataPlotting_v4.m), so the
> thicknesses behind the rat perineurium fit are recoverable only by re-measuring the traces.

The values in the file's `thkperi_um` are counted and set against the number of rat perineurial
rings measured from the traces of version 5.

In [ ]:
om = pelot.read_mat("sparc-60", "files/code/RatVNMorphology_OutputMetrics.mat")
print("variables:", sorted(k for k in om if not k.startswith("__")))
print("samples in the file:", len(om["sub_sam"]))
thk = np.concatenate([np.atleast_1d(x).ravel().astype(float) for x in np.atleast_1d(om["thkperi_um"])])
print("thkperi_um: %d entries, %d values:" % (len(np.atleast_1d(om["thkperi_um"])), len(thk)), np.round(thk, 3))

rows = []
for s in om["sub_sam"]:
    sub, sam = (int(x) for x in s[1:].split("-"))
    r = pelot.analyse_rat(sub, sam)
    rows.append(dict(sample=s, rings=len(r["fasc"]), thk_mean_um=r["fasc"].thk.mean() if "thk" in r["fasc"] else np.nan))
rat = pd.DataFrame(rows)
print(rat.round(3).to_string(index=False))
print("perineurial rings measured from the traces:", rat.rings.sum())

## Earlier finding 4

> samples.xlsx describes the subdiaphragmatic samples as 'subdiaphragmatic vagus nerve' without
> saying they are the anterior trunk, which only the readme and the code's file names give. R17-1
> has a NerveMask although it is monofascicular, and nothing says whether it is meant to be used.

The sample sheet's locations, the places that say "anterior", the NerveMask files and R17-1's
rings are listed.

In [ ]:
smp = pelot.read_excel("sparc-60", "files/samples.xlsx")
loc = smp["specimen anatomical location"].str.split(";").str[0]
print(pd.crosstab(loc, "samples"))
print("rows of samples.xlsx that say anterior:", int(smp.astype(str).apply(lambda c: c.str.contains("nterior")).any(axis=1).sum()))

fs = pelot.files("sparc-60")
print("files whose name says Anterior:", sorted(p.split("/")[-1] for p in fs if "Anterior" in p))
readme = pelot.read_bytes("sparc-60", "readme.md").decode("utf8")
print("readme:", [l.strip()[:160] for l in readme.splitlines() if "anterior" in l.lower()])

nerve = sorted(p for p in fs if p.endswith("NerveMask.tif"))
print("NerveMask files:", [p.split("/")[-1] for p in nerve])
for sub, sam in [(16, 1), (17, 1), (22, 1), (23, 1)]:
    r = pelot.analyse_rat(sub, sam)
    print("R%d-%d: %d perineurial ring(s), NerveMask effective diameter %.1f um" % (sub, sam, len(r["fasc"]), r["ct_d"]))
texts = {p: pelot.read_bytes("sparc-60", p).decode("utf8", "replace")
         for p in ["readme.md", "files/code/README.txt", "files/derivative/README.txt", "files/docs/Readme", "files/protocol/Readme"]}
print("text files that mention NerveMask:", [p for p, t in texts.items() if "NerveMask" in t])